# 알약 검수 파이프라인 서버 (Colab)

**사전 준비 (Google Drive에 업로드):**
```
MyDrive/pill_project/
  inspection_pipeline/   ← pipeline.py, server.py 등 파이프라인 파일들
  model/
    rtmdet_single_pill_aug_300.py
    best_coco_bbox_mAP_50_epoch_273.pth
```

**런타임 → 런타임 유형 변경 → GPU (T4) 선택 필수**

In [ ]:
import os
for v in ['3.10', '3.11', '3.9']:
    path = f'/usr/bin/python{v}'
    if os.path.exists(path):
        print(f"사용 가능: {path}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# [1] GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 - 런타임을 GPU로 변경하세요')

In [ ]:
import sys
!{sys.executable} -m pip install -U pip setuptools wheel
print(f"pip path: {sys.executable}")

In [ ]:
import sys
!{sys.executable} -m pip --version

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y torchaudio || true
!{sys.executable} -m pip install -q torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121
!{sys.executable} -m pip install -q mmengine
!{sys.executable} -m pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3/index.html
!{sys.executable} -m pip install -q mmdet
!{sys.executable} -m pip install -q fastapi uvicorn python-multipart easyocr scipy
!{sys.executable} -m pip install -q pyngrok
print("설치 완료")

In [ ]:
import subprocess
from pyngrok import ngrok
ngrok.kill()
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
print("종료 완료")

In [ ]:
# [3] Google Drive 마운트 및 파일 복사
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/pill_project'
dst = '/content/pill_project'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('복사 완료:')
!ls /content/pill_project/inspection_pipeline/

In [ ]:
# [4] 모델 경로 환경변수 설정
import os
os.environ['PILL_CONFIG']     = '/content/pill_project/model/rtmdet_single_pill_aug_300.py'
os.environ['PILL_CHECKPOINT'] = '/content/pill_project/model/best_coco_bbox_mAP_50_epoch_273.pth'
print('설정 완료')

In [ ]:
init_path = '/usr/local/lib/python3.12/dist-packages/mmdet/__init__.py'
with open(init_path, 'r') as f:
    c = f.read()
c = c.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
with open(init_path, 'w') as f:
    f.write(c)
print("패치 완료")

In [ ]:
import subprocess, threading, time, os, sys
from pyngrok import ngrok

NGROK_TOKEN = '3DZ57MhkCpvpLDb5ddsTEaeFRe2_5cAG4pKWJ5LhgFwrjsNap'
ngrok.set_auth_token(NGROK_TOKEN)

env = os.environ.copy()
env.pop('MPLBACKEND', None)

log_file = open('/tmp/server.log', 'w')

def run_server():
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'server:app',
         '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/pill_project/inspection_pipeline',
        stdout=log_file, stderr=log_file,
        env=env
    )

threading.Thread(target=run_server, daemon=True).start()
time.sleep(8)
tunnel = ngrok.connect(8000)
SERVER_URL = tunnel.public_url
print(f'서버 주소: {SERVER_URL}')
print(f'분석 엔드포인트: {SERVER_URL}/analyze')

In [ ]:
import subprocess
result = subprocess.run(
    ['find', '/tmp', '-name', '*.mp4'],
    capture_output=True, text=True
)
print(result.stdout)